# 02 — BLASTp sequence-search baseline

**Study:** Evaluating Protein Language Models for Function-Based Screening of Heme-Binding Proteins Against Traditional Sequence-Search Methods

**Question for this notebook:** Which of our 50 test proteins have detectable sequence similarity to at least one of the 30 heme-binding reference proteins?

- Search setting: `blastp`, database of 30 reference sequences, report matches with `E-value <= 10`, maximum one high-scoring segment per query–reference pair.
- Main measurements: best bit score per test protein, E-value, percentage identity, alignment length, query coverage.
- Significant-hit reporting rule: **E-value <= 0.01**.
- We also repeat the search after using only the first **1,022 residues of NOS3 (P29474)**, matching the ESM-2 run.
- Run the code cells in order in Google Colab. No model predictions are trained.

In [ ]:
# Import the packages used by the BLASTp experiment.
import shutil
import subprocess
import pandas as pd
from sklearn.metrics import roc_auc_score

# Install NCBI BLAST+ if the current Colab session does not have it.
if shutil.which("blastp") is None or shutil.which("makeblastdb") is None:
    print("Installing NCBI BLAST+ ...")
    subprocess.run(["apt-get", "-qq", "update"], check=True, stdout=subprocess.DEVNULL)
    subprocess.run(["apt-get", "-qq", "install", "-y", "ncbi-blast+"], check=True, stdout=subprocess.DEVNULL)

# Record the BLASTp version used in this execution.
version = subprocess.run(["blastp", "-version"], check=True, capture_output=True, text=True)
print(version.stdout.splitlines()[0])

In [ ]:
# Find our 80-protein dataset beside a cloned GitHub repository, or upload it in Colab.
from pathlib import Path
import pandas as pd
from IPython.display import display

# First check common locations for the dataset.
candidates = [
    Path("../data/master_dataset_80_proteins.csv"),
    Path("data/master_dataset_80_proteins.csv"),
    Path("master_dataset_80_proteins.csv"),
]

# Use the first existing file, if available.
existing = next((path for path in candidates if path.exists()), None)

# Otherwise request a file upload in Google Colab.
if existing is None:
    from google.colab import files
    print("Upload master_dataset_80_proteins.csv")
    uploaded = files.upload()
    if "master_dataset_80_proteins.csv" not in uploaded:
        raise ValueError("Please select master_dataset_80_proteins.csv")
    existing = Path("master_dataset_80_proteins.csv")

# Read the data and make sure the records match our study design.
df = pd.read_csv(existing).reset_index(drop=True)
assert len(df) == 80, "Expected 80 protein records."
assert df["record_id"].is_unique, "Protein record IDs must be unique."
assert df["amino_acid_sequence"].notna().all(), "Missing amino-acid sequences."
assert df["dataset_group"].value_counts().to_dict() == {
    "REFERENCE": 30,
    "AI_DESIGNED_POSITIVE": 20,
    "NATURAL_POSITIVE": 10,
    "NEGATIVE_CONTROL": 20,
}, "Unexpected group sizes."
print("Loaded", len(df), "proteins from", existing)
print(df["dataset_group"].value_counts().to_string())

In [ ]:
# Define the same tabular BLASTp fields used in the pilot.
blast_columns = [
    "query_id", "reference_id", "percent_identity", "alignment_length",
    "query_length", "query_start", "query_end", "evalue", "bitscore",
]

# Write one FASTA record per protein.
def write_fasta(table, filename):
    with open(filename, "w") as fasta:
        for _, protein in table.iterrows():
            fasta.write(f">{protein['record_id']}\n{protein['amino_acid_sequence']}\n")

# Compare all 50 test proteins to our 30 references and return the BLASTp results.
def run_blast(reference_table, test_table, prefix):
    # Save reference sequences and test sequences separately.
    reference_fasta = f"{prefix}_references.fasta"
    test_fasta = f"{prefix}_tests.fasta"
    write_fasta(reference_table, reference_fasta)
    write_fasta(test_table, test_fasta)

    # Prepare a BLASTp search database containing only our 30 references.
    subprocess.run(
        ["makeblastdb", "-in", reference_fasta, "-dbtype", "prot"],
        check=True, stdout=subprocess.DEVNULL
    )

    # Search all 50 proteins, preserving weak hits up to E-value 10.
    raw_file = f"{prefix}_all_hits.tsv"
    subprocess.run([
        "blastp", "-query", test_fasta, "-db", reference_fasta,
        "-evalue", "10", "-max_hsps", "1",
        "-outfmt", "6 qseqid sseqid pident length qlen qstart qend evalue bitscore",
        "-out", raw_file,
    ], check=True)

    # Read every BLASTp hit, including weak matches.
    try:
        hits = pd.read_csv(raw_file, sep="\t", names=blast_columns, header=None)
    except pd.errors.EmptyDataError:
        hits = pd.DataFrame(columns=blast_columns)

    # Calculate how much of each test sequence was aligned.
    if len(hits):
        hits["query_coverage_percent"] = (
            (hits["query_end"] - hits["query_start"] + 1)
            / hits["query_length"] * 100
        ).round(2)
    else:
        hits["query_coverage_percent"] = pd.Series(dtype=float)

    # Select each test protein's strongest bit-score alignment.
    top = (
        hits.sort_values(["bitscore", "evalue"], ascending=[False, True])
        .drop_duplicates("query_id")
    )

    # Keep all 50 test proteins, even if one has no reported BLASTp hit.
    result = test_table[[
        "record_id", "protein_name", "dataset_group", "heme_binding_label"
    ]].merge(top, left_on="record_id", right_on="query_id", how="left")

    # Save original raw hits and best-hit summary for later research analysis.
    hits.to_csv(f"{prefix}_all_hits.csv", index=False)
    result.to_csv(f"{prefix}_results.csv", index=False)
    return hits, result

In [ ]:
# Separate reference proteins from proteins being tested.
references = df[df["dataset_group"] == "REFERENCE"].copy()
tests = df[df["dataset_group"] != "REFERENCE"].copy()

# Run BLASTp first with the original, full-length reference sequences.
original_hits, original_results = run_blast(references, tests, "blastp_original")

# Preserve the original filenames used in our first experiment.
original_hits.to_csv("blastp_all_hits.csv", index=False)
original_results.to_csv("blastp_top_matches.csv", index=False)

# Count how many proteins have significant matches at our E-value threshold.
print("BLASTp significant matches at E <= 0.01:")
for group, block in original_results.groupby("dataset_group"):
    count = (block["evalue"].fillna(float("inf")) <= 0.01).sum()
    print(f"  {group}: {count}/{len(block)}")

# Display the key alignment measurements for each test protein.
display(original_results[[
    "protein_name", "dataset_group", "reference_id", "percent_identity",
    "query_coverage_percent", "evalue", "bitscore",
]])

In [ ]:
# Copy the reference table so the original protein sequences remain unchanged.
matched_references = references.copy()

# ESM-2 used only the first 1,022 residues of the long NOS3 reference P29474.
mask = matched_references["record_id"] == "P29474"
assert mask.sum() == 1, "Expected one NOS3 reference."
matched_references.loc[mask, "amino_acid_sequence"] = (
    matched_references.loc[mask, "amino_acid_sequence"].str[:1022]
)
print("NOS3 length in matched reference:",
      len(matched_references.loc[mask, "amino_acid_sequence"].iloc[0]))

# Rerun BLASTp against the matched-length reference collection.
matched_hits, matched_results = run_blast(
    matched_references, tests, "blastp_matched_length"
)

# Print the original and matched AUC values based on each protein's best bit score.
print("\nBLASTp AUC: original vs matched-length reference")
for group in ["ALL", "AI_DESIGNED_POSITIVE", "NATURAL_POSITIVE"]:
    if group == "ALL":
        original_subset = original_results
        matched_subset = matched_results
    else:
        keep = [group, "NEGATIVE_CONTROL"]
        original_subset = original_results[original_results["dataset_group"].isin(keep)]
        matched_subset = matched_results[matched_results["dataset_group"].isin(keep)]

    # Assign label 1 to heme binders and label 0 to controls.
    y_original = original_subset["heme_binding_label"].str.lower().eq("positive").astype(int)
    y_matched = matched_subset["heme_binding_label"].str.lower().eq("positive").astype(int)
    old_auc = roc_auc_score(y_original, original_subset["bitscore"].fillna(0))
    new_auc = roc_auc_score(y_matched, matched_subset["bitscore"].fillna(0))
    print(f"  {group}: original {old_auc:.3f}; matched {new_auc:.3f}")

# List natural positives with no significant BLASTp match.
missed = matched_results[
    (matched_results["dataset_group"] == "NATURAL_POSITIVE") &
    (matched_results["evalue"].fillna(float("inf")) > 0.01)
]
print("\nNatural positives without E <= 0.01 hit:", missed["protein_name"].tolist())

display(matched_results[["protein_name", "dataset_group", "bitscore", "evalue"]])

In [ ]:
# Download the exact files required for the next analysis notebook.
from google.colab import files

# The matched-length BLASTp file is the one used for the final AUC comparison.
files.download("blastp_matched_length_results.csv")

# Preserve the full hit list to audit alignments later.
files.download("blastp_matched_length_all_hits.csv")

# Also preserve the original full-length results for the truncation comparison.
files.download("blastp_top_matches.csv")
files.download("blastp_all_hits.csv")

### Outputs

- `blastp_all_hits.csv` and `blastp_top_matches.csv` — first full-length-reference run.
- `blastp_matched_length_all_hits.csv` and `blastp_matched_length_results.csv` — matched NOS3 reference run.

**Observed in the original pilot:** Significant matches at E <= 0.01: designed 20/20, natural 3/10, controls 2/20. Matched-length AUC: all 0.905, designed-vs-controls 1.000, natural-vs-controls 0.715.